# User Behavior Analysis

<!-- course-learning:start -->
**学习重点**

- [先查字段，明确当前数据能支持哪些行为指标。](#section-2-2)
- [区分用户占比与帖子贡献占比，为每个指标写清分母。](#section-3-3)
- [比较帖子等权与用户等权，理解高发帖用户对结果的影响。](#section-4-2)
- [回到真实文本核验关键词规则，区分文字命中与真实行为、心理状态。](#section-5-2)
- [分别解释城市记录总量与每位已观测用户的平均记录数。](#section-6)

**标记说明：** **[重点]** 表示应理解并能做简单修改；**[练习]** 表示动手作答；**[拓展]** 表示进一步学习的方法。未加标记的内容用于准备、解释或查阅。

标记表示学习优先级。代码格依次运行，以建立后续所需的变量；拓展代码格可先观察结果，再深入学习原理。
<!-- course-learning:end -->

<!-- course-toc:start -->
<a id="course-toc"></a>
**目录**

- [1 分析哪些行为？](#section-1) **[重点]**
- [2 数据能告诉我们什么？](#section-2)
  - [2.1 环境和真实输入](#section-2-1)
  - [2.2 能不能分析转发、@提及或点赞？先查字段](#section-2-2) **[重点]**
- [3 哪些用户贡献了记录？](#section-3) **[重点]**
  - [3.1 从帖子表变成用户表](#section-3-1) **[重点]**
  - [3.2 均值和中位数分别回答什么？](#section-3-2)
  - [3.3 用户占比和记录贡献占比的分母不同](#section-3-3) **[重点]**
- [4 用户更常谈哪些内容？](#section-4) **[重点]**
  - [4.1 把内容问题变成可复核的关键词规则](#section-4-1) **[重点]**
  - [4.2 同一词表，帖子等权与用户等权](#section-4-2) **[重点]**
  - [4.3 为观察用户内容结构，明确记录门槛](#section-4-3) **[重点]**
- [5 公开表达中的情绪词](#section-5) **[重点]**
  - [5.1 四类可解释规则](#section-5-1)
  - [5.2 回到文本，看看规则命中了什么](#section-5-2) **[重点]**
  - [5.3 文件里的源模型标签怎样使用？](#section-5-3) **[拓展]**
- [6 城市汇总：数量与每用户记录数](#section-6) **[重点]**
- [7 开放问题](#section-7) **[练习]**
  - [7.1 开放问题：怎样定义“记录较多的用户”？](#section-7-1) **[练习]**
  - [7.2 开放问题：“更常谈饮食”会不会只是词表定义的结果？](#section-7-2) **[练习]**
  - [7.3 开放问题：怎样定义“负向表达更多”？](#section-7-3) **[练习]**
- [8 分析流程](#section-8)
<!-- course-toc:end -->

用数据回答：哪些用户贡献了记录？他们更常谈什么？公开文本中出现了哪些情绪词？

输入为 `tutorials/data/posts_two_weeks.parquet` 中四川的可用记录。
源记录不是平台全部发帖，也不是完整用户历史。
城市作为来源标签分组；日期用于确定两周窗口和计算观测到发帖的天数。

<a id="section-1"></a>
## 1 分析哪些行为？ **[重点]**

分析对象是已采集记录中的发帖与公开表达，以及这些记录在城市来源标签上的分布。

| 维度 | 具体观察 | 指标解释 |
|---|---|---|
| 发帖数量 | 两周记录数、出现日期数、记录贡献比例 | 窗口内采集到的发帖记录量 |
| 发帖内容 | 工作、学习、饮食、休闲、出行话题、公共事务词的命中比例 | 记录中出现这些话题词的比例 |
| 公开情绪表达 | 正向词、负向词、同时命中、未命中 | 公开文本中的情绪词命中情况 |
| 社交互动 | 转发、@提及、点赞、评论 | 先检查是否有可验证的结构化字段 |
| 城市汇总 | 各来源城市记录数、用户数、每用户平均记录数 | 城市标签下采集记录的分布 |

谈论饮食不一定表示喜欢饮食，提到出行不等于实际出行，情绪词也不等于用户的心理状态。
每个指标都需要明确分析单位、分母和解释范围。

<a id="data"></a>
<a id="section-2"></a>
## 2 数据能告诉我们什么？

<a id="section-2-1"></a>
### 2.1 环境和真实输入

在 VS Code 打开整个项目，选择课程 Python 内核（Conda 或 venv）。本教程独立初始化，
不依赖其他 Notebook 的变量，只用现有 Pandas、Matplotlib 与 PyArrow 依赖。
输出表格与图显示在 Notebook 中。

In [ ]:
# Path 处理文件路径；用 / 拼接目录，不必写本机绝对路径。
from pathlib import Path
# sys 提供当前解释器路径和环境目录，用来检查 Notebook 内核。
import sys
# re 是正则表达式标准库；re.escape 把关键词中的特殊符号转为普通字符。
import re
# NumPy 处理数值数组、分位数和布尔条件；np 是常用别名。
import numpy as np
# Pandas 用 Series / DataFrame 处理带行列标签的表格；pd 是别名。
import pandas as pd
# PyArrow 读取 Parquet 的模式、行组和元数据；pq 是别名。
import pyarrow.parquet as pq
# pyplot 创建图形和绘图区，并控制图形显示；plt 是别名。
import matplotlib.pyplot as plt
# font_manager 查询本机字体，便于选择能显示中文的字体。
from matplotlib import font_manager
# PercentFormatter 把 0–1 的比例显示为 0%–100%，不改变原数据。
from matplotlib.ticker import PercentFormatter
# display 在 Notebook 输出区显示表格或图形对象。
from IPython.display import display

# assert 检查当前解释器属于 Conda 或 venv；失败时停止，提示选择安装了课程依赖的内核。
assert (Path(sys.prefix) / 'conda-meta').exists() or sys.prefix != sys.base_prefix, '请选择安装课程依赖的 Conda 或 venv 内核'
# 根据数据所在位置选择相对目录，兼容从 tutorials 或项目根目录启动。
TUTORIAL_DIR = Path('.') if Path('data/posts_two_weeks.parquet').is_file() else Path('tutorials')
# 拼接同一份两周 Parquet 的输入路径，后面的分析都使用它。
DATA = TUTORIAL_DIR / 'data/posts_two_weeks.parquet'
# 确认输入文件存在；路径不正确时先修正启动目录或补齐课堂数据。
assert DATA.is_file(), '请从 tutorials 或项目根目录启动内核，并保留 tutorials/data'
# resolve() 转为完整路径，parent 取得 tutorials 的上一级项目目录。
ROOT = TUTORIAL_DIR.resolve().parent
# 只读取 Parquet 模式中的字段名，用于检查文件是否真的包含互动指标。
source_columns = pq.ParquetFile(DATA).schema_arrow.names
# 按 columns 只读取需要的字段，filters 只保留四川可用记录；得到 Pandas DataFrame。
raw = pd.read_parquet(DATA,
    columns=['post_id', 'user_id', 'date', 'city', 'text',  # 只读取列清单中的字段，减少不必要的读取和内存占用。
             'sentiment_label_source', 'score_valid'],
    filters=[('province_code', '==', '51'), ('usable_text', '==', True)])  # 过滤条件为四川省编码 51 且文本可用；同一组条件按“并且”组合。
# 复制读入的表；后面新增列写入 posts，不修改 raw。
posts = raw.copy()

# 收集本机已安装字体名，不假设每台电脑都有相同的中文字体。
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
# next() 取候选列表中第一个已安装字体；没有匹配时返回 None。
CJK_FONT = next((name for name in ['Noto Sans CJK SC', 'Arial Unicode MS',
    'Microsoft YaHei', 'SimHei', 'Songti SC', 'Heiti TC', 'Noto Sans CJK JP']
    if name in available_fonts), None)
# 仅在找到可用中文字体时设置字体；否则图题使用英文回退。
if CJK_FONT:
    # 把找到的中文字体设为 Matplotlib 默认字体。
    plt.rcParams['font.family'] = CJK_FONT
# 设置后续图形的默认字号、尺寸、负号与边框显示，不改变统计数据。
plt.rcParams.update({'font.size': 11, 'figure.figsize': (9, 4),
    'axes.unicode_minus': False, 'axes.spines.top': False, 'axes.spines.right': False})
# 限制表格最多显示的行数，超出部分折叠；不删数据。
pd.set_option('display.max_rows', 10)
# 限制表格最多显示的列数，便于阅读；不删字段。
pd.set_option('display.max_columns', 8)
# 限制单元格显示宽度，长文本省略展示；正文仍完整保留。
pd.set_option('display.max_colwidth', 90)

def label(chinese, english):
    """输入：中文、英文字符串；输出：适合当前字体的图题或轴标签。依赖已检查的 CJK_FONT。"""
    # 找到中文字体时返回中文，否则返回英文；函数只选择文字，不改变数据。
    return chinese if CJK_FONT else english

# 两周观察窗口共 14 天，作为用户出现日期比例的统一分母。
WINDOW_DAYS = 14
# 打印内核实际使用的 Python 路径，确认选中了课程环境。
print('Pandas：', pd.__version__, 'Python：', sys.executable)
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('帖子记录数：', len(posts), '不同用户数：', posts['user_id'].nunique(),
      '出现日期数：', posts['date'].nunique())
# 显示结果的前几行，观察索引、列名和数值；head 不改变完整统计结果。
display(posts[['date', 'city', 'text']].head(3))

**怎么看结果：** 默认范围有 54,609 条可用记录、36,463 个匿名用户，窗口为 2022-08-12 至 08-25。
`post_id` 是帖子主键，`user_id` 用于把同一用户的记录组合起来。
用户不是逐行独立的：同一个人的多条帖子可能相关，不能在统计推断时直接当成独立样本。

<a id="section-2-2"></a>
### 2.2 能不能分析转发、@提及或点赞？先查字段 **[重点]**

一个平台可能支持很多行为，但一个数据文件不一定保留这些行为的字段。
本文件没有独立的原创/转发标记、转发父帖 ID、被提及账号列表或互动数量。
文本来自预处理的 `text_clean`，随后又做了脱敏；即使某个文本模式看不见，也不能断言原帖没有对应行为。

In [ ]:
# 列出常见的转发、提及、点赞和评论字段，随后检查是否存在；不会补造这些指标。
interaction_fields = ['is_repost', 'retweeted_status_id', 'mention_ids',
                       'reposts_count', 'likes_count', 'comments_count']
# 列表推导逐个检查字段是否存在，形成“字段名—存在与否”的审计表。
field_audit = pd.DataFrame({
    '示意字段名': interaction_fields,
    '当前文件是否存在': [name in source_columns for name in interaction_fields],
})
# 在 Notebook 输出区显示本步的结果表。
display(field_audit)
# 统计正文中可见字符或字面词的出现次数；这不等于结构化的提及或转发次数。
text_audit = pd.Series({
    '正文含 @ 字符的记录数': posts['text'].str.contains('@', regex=False, na=False).sum(),
    '正文含 [提及] 掩码的记录数': posts['text'].str.contains('[提及]', regex=False, na=False).sum(),
    '正文含“转发”一词的记录数': posts['text'].str.contains('转发', regex=False, na=False).sum(),
}, name='可见文本计数')
# 将 Series 转成单列表格并显示，便于一起查看行标签与统计值。
display(text_audit.to_frame())

**怎么看结果：** 本范围里没有可见的 `@` 或 `[提及]`；有 21 条正文包含“转发”一词。
**21 不是转发帖数，0 也不是用户从未提及别人。**
提到“转发”可以是在讨论这个动作，不能替代平台转发元数据。
将来有可验证的动作类型、被提及账号 ID 和转发父帖 ID，才适合分析这些互动或关系。
本教程因此把记录统一称为“帖子记录”，不把它们全部判为原创，也不估计传播影响力。
字段说明见项目 `tutorials/data/README.md`，脱敏过程见 `scripts/prepare_two_weeks.py`。

<a id="activity"></a>
<a id="section-3"></a>
## 3 哪些用户贡献了记录？ **[重点]**

<a id="section-3-1"></a>
### 3.1 从帖子表变成用户表 **[重点]**

先定义指标，再选聚合方式。每个用户一行可以方便地比较用户之间的差异。

| 字段 | 定义 | 分析单位 |
|---|---|---|
| `n_posts` | 该用户在窗口内的采集记录数 | 用户 |
| `observed_days` | 有采集记录的不同日期数 | 用户 |
| `observed_day_share` | 观测日期数 / 14 个日历日 | 用户 |
| `mean_text_length` | 该用户已采集正文的平均字符数 | 用户 |

`observed_day_share` 描述窗口内观测到的发帖日期覆盖，不等于每天使用微博的概率。
字符数也不是词数。没有采集记录的日期，不能直接解释为用户没有使用微博。

In [ ]:
# str.len() 逐条计算正文字符数；中文字符计为字符，不是词数或字节数。
posts['text_length'] = posts['text'].str.len()
# 按用户 ID 聚合成一人一行，统计记录数、不同发帖日期数和平均正文长度。
users = posts.groupby('user_id').agg(
    n_posts=('post_id', 'size'),  # size 统计组内行数，不是不同用户数。
    observed_days=('date', 'nunique'),  # 统计出现过记录的不同日期数。
    mean_text_length=('text_length', 'mean'),  # 求组内记录正文长度的平均值。
)
# 用出现过记录的日期数除以 14 天；它描述采集窗口内的观测，不是完整活跃率。
users['observed_day_share'] = users['observed_days'] / WINDOW_DAYS
# 显示结果的前几行，观察索引、列名和数值；head 不改变完整统计结果。
display(users.head(3))
# describe() 给出计数、均值、标准差和分位数，帮助检查指标分布。
display(users[['n_posts', 'observed_days', 'mean_text_length']].describe())
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('用户表累计记录数：', users['n_posts'].sum(), '原帖子表记录数：', len(posts))

<a id="section-3-2"></a>
### 3.2 均值和中位数分别回答什么？

均值反映所有用户的记录总量除以人数；中位数反映排序后处于中间位置的用户。
若少数用户有很多记录，均值可能大于多数用户的记录数。
先看分位数和记录数分布，再决定怎样描述“典型用户”。

In [ ]:
# 汇总用户记录数的均值、中位数、分位数与最大值，用来观察偏态与长尾。
summary = pd.Series({
    '用户数': len(users),
    '每用户平均记录数': users['n_posts'].mean(),
    '每用户记录数中位数': users['n_posts'].median(),
    '记录数75%分位数': users['n_posts'].quantile(0.75),
    '记录数95%分位数': users['n_posts'].quantile(0.95),
    '最大记录数': users['n_posts'].max(),
}, name='统计值')
# 将 Series 转成单列表格并显示，便于一起查看行标签与统计值。
display(summary.to_frame())
# value_counts() 统计各值的频数；dropna=False 时也显示缺失值的频数。
display(users['n_posts'].value_counts().sort_index().head(8).rename('用户数').to_frame())

**怎么看结果：** 当前窗口中多数用户只被观测到一条帖子，记录数中位数为 1。
这描述样本的稀疏程度，不能据此把他们称为平台上的“低活跃用户”。

<a id="section-3-3"></a>
### 3.3 用户占比和记录贡献占比的分母不同 **[重点]**

将用户按采集记录数划为互斥组：1 条、2 条、3–4 条、5 条及以上。
一种比例数人，另一种比例数帖；两者都要报告。

In [ ]:
# cut 默认右闭区间：1、2、3–4、5+；np.inf 让最后一组包含所有更大的记录数。
groups = pd.cut(users['n_posts'], bins=[0, 1, 2, 4, np.inf],
                labels=['1', '2', '3–4', '5+'])  # 把 Plotly 字段名换成坐标与图例中可读的名称。
# 按记录数组统计人数与记录贡献；observed=False 保留没有用户的分类组。
group_summary = users.assign(record_group=groups).groupby(
    'record_group', observed=False).agg(n_users=('n_posts', 'size'), n_posts=('n_posts', 'sum'))
# 人数除以全体已观测用户数；分母的单位是用户。
# 重点：用户占比的分母是全部已观测用户数 len(users)，统计单位是用户。
group_summary['user_share'] = group_summary['n_users'] / len(users)
# 组内用户的记录总数除以全体帖子记录数；分母的单位是帖子。
# 重点：记录贡献占比的分母是全部帖子数 len(posts)，不能沿用用户比例的分母。
group_summary['post_share'] = group_summary['n_posts'] / len(posts)
# 在 Notebook 输出区显示本步的结果表。
display(group_summary)
# 选取两种比例并重命名图例；两列分母不同，不能混为同一个“占比”。
plot_groups = group_summary[['user_share', 'post_share']].rename(columns={
    'user_share': label('用户占比', 'User share'), 'post_share': label('记录贡献占比', 'Post share')})
# 按类别画柱形图；多列默认并列，stacked=True 才堆叠，rot 控制类别文字旋转。
ax = plot_groups.plot.bar(rot=0, color=['#2563a6', '#c56a28'])
# 为当前绘图区设置图题、坐标名称和单位；label() 根据字体选择中文或英文文字。
ax.set(title=label('各记录数组：人数与记录贡献', 'Users and post contribution by record count'),
       xlabel=label('窗口采集记录数组', 'Collected posts per user in window'),  # 横轴名称及单位。
       ylabel=label('比例', 'Proportion'))  # 纵轴名称及单位。
# xmax=1 把原值 1 显示为 100%；只改刻度标签，不把数据乘以 100。
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
# 加浅色网格便于读取数值；axis 指定方向，alpha 控制透明度。
ax.grid(axis='y', alpha=0.2)
# 自动调整子图间距，让坐标文字、标题与图例尽量不重叠。
plt.tight_layout()
# 在 Notebook 输出区显示当前图形。
plt.show()

**怎么看结果：** 一组人数较少，也可能贡献相当一部分帖子。
分组门槛是分析者选的定义，不是自然存在的用户类型。下面的开放问题会比较不同门槛。

<a id="content"></a>
<a id="section-4"></a>
## 4 用户更常谈哪些内容？ **[重点]**

<a id="section-4-1"></a>
### 4.1 把内容问题变成可复核的关键词规则 **[重点]**

不直接给用户贴“职业”“爱好”标签，而是先统计记录中的词命中。
词表沿用主课的六个讨论维度，便于在教程和实验之间切换。
每条帖子可以同时命中多类，因此六类比例相加不一定等于 1。

In [ ]:
# 定义主题到关键词列表的字典；类别允许重叠，这是词表规则而不是训练出的主题模型。
TOPICS = {
    '工作': ['工作', '上班', '加班', '公司', '同事', '办公'],
    '学习': ['学习', '考试', '作业', '学校', '大学', '论文'],
    '饮食': ['吃饭', '好吃', '火锅', '奶茶', '美食', '早餐', '午饭', '晚饭'],
    '休闲': ['电影', '音乐', '游戏', '演唱会', '追剧', '公园'],
    '出行话题': ['旅行', '旅游', '出差', '高铁', '航班', '地铁', '公交'],
    '公共事务': ['停电', '限电', '停水', '疫情', '政府', '社区', '防疫', '核酸'],
}

def keyword_hit(text_series, words):
    # 转义词中的正则字符；| 表示命中任意一个词。
    """输入：文本 Series 和非空关键词列表；输出：逐条是否命中的布尔 Series，保留输入索引。"""
    # 先用 re.escape 转义每个词，再以 | 连接；正则中的 | 表示匹配任一关键词。
    pattern = '|'.join(re.escape(word) for word in words)
    # str.contains 返回与输入索引一致的布尔 Series；na=False 让缺失文本返回 False。
    return text_series.str.contains(pattern, regex=True, na=False)

# 每一列是一类主题，每一行对应一条帖子；字典推导逐类执行关键词检测，保留帖子索引。
topic_flags = pd.DataFrame({name: keyword_hit(posts['text'], words)
                            for name, words in TOPICS.items()}, index=posts.index)
# 在 Notebook 输出区显示本步的结果表。
display(pd.DataFrame({'主题': TOPICS.keys(), '词表': ['、'.join(w) for w in TOPICS.values()]}))
# 布尔列的 sum 是命中记录数，mean 是命中记录数占全部记录的比例。
topic_summary = pd.DataFrame({'命中帖子数': topic_flags.sum(), '帖子命中比例': topic_flags.mean()})
# 在 Notebook 输出区显示本步的结果表。
display(topic_summary.sort_values('命中帖子数', ascending=False))
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('六类中至少命中一类的帖子比例：', topic_flags.any(axis=1).mean())
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('六类全部未命中的帖子数：', (~topic_flags.any(axis=1)).sum())

“出行话题”只是讨论内容，未使用坐标判断真实出行。
`str.contains` 是子串匹配，不是中文分词或语义模型。例如“公司”也可能出现在转述、新闻或广告中。
多标签允许重叠，但未命中的记录仍必须留在相应分母中。

<a id="section-4-2"></a>
### 4.2 同一词表，帖子等权与用户等权 **[重点]**

对所有帖子求均值，每条帖子权重相同，高记录用户会贡献更多。
先对每个用户求命中比例，再对用户求均值，每位已观测用户权重相同。
二者都合理，但回答不同的问题。

In [ ]:
# 按 user_id 对布尔命中取均值，得到每位用户各类话题的记录比例。
# 重点：先算各用户的命中比例，再对用户求均值，才是用户等权；直接对帖子求均值是帖子等权。
topic_user_share = topic_flags.groupby(posts['user_id']).mean()
# 按用户索引左连接主题比例；add_prefix 为新列加前缀，避免覆盖已有用户指标。
users = users.join(topic_user_share.add_prefix('topic_share_'))
# 分别对所有帖子和所有用户的比例取均值，比较帖子等权与用户等权结果。
topic_weighting = pd.DataFrame({
    'post_weighted': topic_flags.mean(),  # 每条帖子等权，高发帖用户会贡献更多记录权重。
    'user_weighted': topic_user_share.mean(),  # 先算各用户比例再求平均，每位已观测用户权重相同。
})
# 在 Notebook 输出区显示本步的结果表。
display(topic_weighting.rename(columns={
    'post_weighted': '帖子等权比例', 'user_weighted': '用户等权比例'}))  # 每条帖子等权，高发帖用户会贡献更多记录权重。
# 按类别画柱形图；多列默认并列，stacked=True 才堆叠，rot 控制类别文字旋转。
ax = topic_weighting.rename(index={
    k: label(k, v) for k, v in zip(TOPICS, ['Work', 'Study', 'Food', 'Leisure', 'Travel topics', 'Public affairs'])
}).rename(columns={'post_weighted': label('帖子等权', 'Posts equally weighted'),
                   'user_weighted': label('用户等权', 'Users equally weighted')}).plot.bar(  # 先算各用户比例再求平均，每位已观测用户权重相同。
    rot=20, color=['#2563a6', '#c56a28'])
# 为当前绘图区设置图题、坐标名称和单位；label() 根据字体选择中文或英文文字。
ax.set(title=label('同一词表，两种观察单位', 'Same rules, different weighting units'),
       xlabel=label('主题词类别', 'Keyword category'), ylabel=label('命中比例', 'Keyword share'))  # 横轴名称及单位。
# xmax=1 把原值 1 显示为 100%；只改刻度标签，不把数据乘以 100。
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
# 自动调整子图间距，让坐标文字、标题与图例尽量不重叠。
plt.tight_layout()
# 在 Notebook 输出区显示当前图形。
plt.show()

**怎么看结果：** 第一列回答“随机取一条已采集帖子，词命中的比例是多少”；
第二列回答“随机取一位已观测用户，其帖子命中比例的平均值是多少”。
一个只有一条记录的用户，某主题比例可能是 0 或 1，不能据此认定稳定偏好。

<a id="section-4-3"></a>
### 4.3 为观察用户内容结构，明确记录门槛 **[重点]**

选取至少有 3 条记录的用户，并报告保留人数。
门槛能减少一条记录就形成 100% 比例的情况，但会改变用户集合；这不是消除采集偏差的方法。

In [ ]:
# 设置最低记录数门槛，用于讨论少量观测带来的不稳定性；这是分析选择。
MIN_POSTS = 3
# ge() 表示大于等于；用布尔条件选出达到记录门槛的用户索引。
eligible_ids = users.index[users['n_posts'].ge(MIN_POSTS)]
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('至少3条记录的用户数：', len(eligible_ids), '占全部已观测用户：', len(eligible_ids) / len(users))
# 显示结果的前几行，观察索引、列名和数值；head 不改变完整统计结果。
display(users.loc[eligible_ids, ['n_posts', 'observed_days'] +
                  ['topic_share_' + name for name in TOPICS]].head(3))
# 取饮食词命中的前三条记录供人工核验；样例不能代替总体统计。
sample_food = posts.loc[topic_flags['饮食'], ['city', 'text']].head(3).copy()
# 只将展示副本截到前 150 个字符，原帖子正文和词表统计不变。
sample_food['text'] = sample_food['text'].str.slice(0, 150)
# 在 Notebook 输出区显示本步的结果表。
display(sample_food)

**怎么看结果：** 当前有 3,295 位用户满足门槛，只覆盖原用户集合的一部分。
饮食示例是前三条可见命中记录，显示文字截到 150 字，完整文本仍用于规则计算。
看几个例子可以发现规则问题，但不能用它们声称词表准确率已经验证。
要描述“更常发什么”，应同时报告词表、用户集合、记录数及抽样核验情况。

<a id="expression"></a>
<a id="section-5"></a>
## 5 公开表达中的情绪词 **[重点]**

<a id="section-5-1"></a>
### 5.1 四类可解释规则

我们使用主课相同的正向与负向词表，只做文本表达的探索。
正负都命中时保留“混合命中”，都未命中时保留“未命中”；未命中不等于中性。
规则不处理所有否定、引用、反讽或多义词，因此不能把结果当成真实心理测量。

In [ ]:
# 给出正向词规则的词表；命中只表示文字含这些词。
POS_WORDS = ['开心', '快乐', '幸福', '舒服', '凉快', '爽', '喜欢', '美好', '期待', '满意']
# 给出负向词规则的词表；词语的上下文、否定和转述仍需要人工检查。
NEG_WORDS = ['难受', '烦躁', '崩溃', '痛苦', '讨厌', '失眠', '焦虑', '中暑', '热死', '受不了', '累', '郁闷', '难熬']
# 逐帖检测是否命中任意正向词，返回与帖子表索引对应的布尔 Series。
positive_hit = keyword_hit(posts['text'], POS_WORDS)
# 逐帖检测任意负向词；同一帖子可以同时命中两类词。
negative_hit = keyword_hit(posts['text'], NEG_WORDS)
# np.select 按条件顺序取第一个匹配结果；先判断混合命中，避免被归入仅负向或仅正向。
# 重点：这里分类的是情绪词命中情况；应检查上下文，不能据此判断用户的心理状态。
emotion_labels = pd.Series(np.select(
    [positive_hit & negative_hit, negative_hit, positive_hit],
    ['混合命中', '仅负向命中', '仅正向命中'], default='未命中'), index=posts.index, name='情绪词规则')
# 固定四类情绪词规则的显示顺序，不按出现次数改变顺序。
emotion_order = ['仅正向命中', '仅负向命中', '混合命中', '未命中']
# 统计四个互斥规则类别的记录数；未出现的类别也补为 0。
emotion_counts = emotion_labels.value_counts().reindex(emotion_order, fill_value=0)
# 把各类记录数与占全部帖子的比例并列；未命中的帖子仍计入分母。
emotion_table = pd.DataFrame({'帖子数': emotion_counts, '占全部帖子比例': emotion_counts / len(posts)})
# 在 Notebook 输出区显示本步的结果表。
display(emotion_table)
# 按类别画柱形图；多列默认并列，stacked=True 才堆叠，rot 控制类别文字旋转。
ax = emotion_table['占全部帖子比例'].rename(index={
    k: label(k, v) for k, v in zip(emotion_order, ['Positive only', 'Negative only', 'Mixed hits', 'No hits'])
}).plot.bar(rot=15, color=['#50896d', '#c56a28', '#8a6eaa', '#b0b8c0'])
# 为当前绘图区设置图题、坐标名称和单位；label() 根据字体选择中文或英文文字。
ax.set(title=label('文本中的情绪词命中情况', 'Emotion keyword matches in text'),
       xlabel=label('规则类别', 'Rule category'), ylabel=label('占全部帖子比例', 'Share of all posts'))  # 横轴名称及单位。
# xmax=1 把原值 1 显示为 100%；只改刻度标签，不把数据乘以 100。
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
# 自动调整子图间距，让坐标文字、标题与图例尽量不重叠。
plt.tight_layout()
# 在 Notebook 输出区显示当前图形。
plt.show()

<a id="section-5-2"></a>
### 5.2 回到文本，看看规则命中了什么 **[重点]**

每类展示最多两条真实记录。观察词表是否抓到了作者自己的表达，还是转述、广告或其他语境。
文本已预处理和脱敏，有些上下文可能丢失；本教程不会尝试恢复被掩码的信息。

In [ ]:
# 给正文与城市展示表添加逐帖规则标签，按原索引对齐。
emotion_examples = posts[['city', 'text']].assign(规则=emotion_labels)
# 按规则类别在每组取前两条记录并复制，用于核验上下文。
emotion_examples = emotion_examples.groupby('规则', sort=False).head(2).copy()
# 截短用于展示的副本，不改原始帖子，也不重新计算标签。
emotion_examples['text'] = emotion_examples['text'].str.slice(0, 150)
# 在 Notebook 输出区显示本步的结果表。
display(emotion_examples[['规则', 'city', 'text']])

<a id="section-5-3"></a>
### 5.3 文件里的源模型标签怎样使用？ **[拓展]**

文件还保留 `sentiment_label_source` 与源分数。数据说明将标签 0 定义为负向、1 定义为正向，
但具体模型版本及本数据上的语义性能没有独立核实。
`score_valid` 检查数值范围和两类分数的和，不检验模型是否正确理解了句子。
所以下面只展示来源标签数量，不把它变成用户幸福感、抑郁程度或“情绪真值”。

In [ ]:
# value_counts() 统计各值的频数；dropna=False 时也显示缺失值的频数。
display(posts['sentiment_label_source'].value_counts(dropna=False).sort_index()
        .rename('来源标签记录数').to_frame())
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('源分数数值有效标记占比：', posts['score_valid'].mean())
# 显示指标的口径说明，解释这一步结果应如何理解。
print('标签分布和数值有效率不是语义准确率。')

<a id="city"></a>
<a id="section-6"></a>
## 6 城市汇总：数量与每用户记录数 **[重点]**

“哪个城市发帖量较多”可以先描述为“哪个来源城市标签下记录较多”。
然后同时查看不同用户数，以及该标签下每位已观测用户的平均记录数。
源城市编码的生成机制未核实，因此城市标签不是经过验证的用户居住地。
这里不分析用户跨城市移动，也不使用坐标。

In [ ]:
# 按城市统计记录数和不同用户数；同一用户在不同城市出现时会分别计入。
city_summary = posts.groupby('city').agg(
    n_posts=('post_id', 'size'), n_users=('user_id', 'nunique'))  # size 统计组内行数，不是不同用户数。
# 各城市记录数除以该城市不同用户数，得到每位已观测用户的平均记录数。
city_summary['posts_per_observed_user'] = city_summary['n_posts'] / city_summary['n_users']
# 显示结果的前几行，观察索引、列名和数值；head 不改变完整统计结果。
display(city_summary.sort_values('n_posts', ascending=False).head(6))
# 按“城市—用户”组合数记录，索引成为两层 MultiIndex。
city_user_records = posts.groupby(['city', 'user_id']).size()
# 按多层索引中的 city 层分组，求每个城市内用户记录数的中位数。
city_summary['median_posts_per_user'] = city_user_records.groupby(level='city').median()

# nlargest(6) 选记录量前六城市，再升序排，便于横条从小到大显示。
count_top = city_summary['n_posts'].nlargest(6).sort_values()
# 示范门槛：避免用极小人数的城市均值直接排名；门槛不是统计显著性标准。
# 只让至少 100 位已观测用户的城市进入均值排名，减少极小分母的影响。
city_comparable = city_summary.loc[city_summary['n_users'].ge(100)]
# 在达到人数门槛的城市中选均值前六项；它与记录总量排名可能是不同城市。
mean_top = city_comparable['posts_per_observed_user'].nlargest(6).sort_values()
# 创建子图（1 行 2 列）；返回 Figure 和 Axes，figsize 为英寸，sharex / sharey 控制共享坐标轴。
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
# 横向柱形图：类别在纵轴、数值在横轴；返回柱子对象，可用于添加数字标签。
axes[0].barh(count_top.index.astype(str), count_top.to_numpy(dtype=float), color='#2563a6')
# 横向柱形图：类别在纵轴、数值在横轴；返回柱子对象，可用于添加数字标签。
axes[1].barh(mean_top.index.astype(str), mean_top.to_numpy(dtype=float), color='#c56a28')
# 为当前绘图区设置图题、坐标名称和单位；label() 根据字体选择中文或英文文字。
axes[0].set(title=label('来源城市：记录数量前六项', 'Top six city labels by post count'),
            xlabel=label('记录数', 'Collected posts'))  # 横轴名称及单位。
# 为当前绘图区设置图题、坐标名称和单位；label() 根据字体选择中文或英文文字。
axes[1].set(title=label('来源城市：每用户平均记录数前六项', 'Top six city labels by mean posts per user'),
            xlabel=label('每位已观测用户的记录数', 'Posts per observed user'))  # 横轴名称及单位。
# 依次操作各个子图；zip 可把绘图区与参数或列名一一配对。
for ax in axes:
    # 设置横轴显示范围，并给柱端标注留出空白；只改变图上的可见范围。
    ax.set_xlim(left=0)
    # 加浅色网格便于读取数值；axis 指定方向，alpha 控制透明度。
    ax.grid(axis='x', alpha=0.2)
# 在 Notebook 输出区显示当前图形。
plt.show()

**怎么看结果：** 两种排名可能不同；记录数量多，可能是用户数多，并非每个用户记录更多。
“每用户记录数”仍受原采集覆盖影响，不能代表城市居民发帖倾向。
同一用户若有多个来源城市标签，会被计入多个城市的用户数，所以各城市用户数相加不一定等于全体去重用户数。
别把记录数、用户数和居民人口数混成同一分母。

<a id="questions"></a>
<a id="section-7"></a>
## 7 开放问题 **[练习]**

每题紧接着一个**真正空白的 Code cell**，可以写自己的分析代码，也可以先讨论再展开答案。
答案只是一个有依据的分析方案，不要求与它得出同样的措辞或采用同样的门槛。
参考代码变量以 `q1_`、`q2_`、`q3_` 开头，可以复制到该题空白格运行，不覆盖上方主要统计表。
点击“运行全部”会运行示例并跳过空白格；不会自动执行折叠答案。

<a id="section-7-1"></a>
### 7.1 开放问题：怎样定义“记录较多的用户”？ **[练习]**

如果把“窗口内至少 3 条记录”与“至少 5 条记录”分别作为门槛，
选出的用户占全部用户多少？贡献了全部记录的多少？
哪个定义更适合你的问题？这些门槛能够说明用户在平台上的真实活跃度吗？

**在下面的空白代码格写自己的比较；参考答案在其后折叠。**

<details><summary>展开 7.1 参考答案与代码</summary><p><strong>参考思路：</strong> 两个门槛都可以用，但必须说明它是“已采集记录数”的分组。 用户占比的分母是全体已观测用户；记录贡献占比的分母是所有采集记录。 提高门槛会缩小用户集合，不能只展示比例而不报告人数。</p><p>当前数据中至少 3 条记录的用户有 3,295 位，至少 5 条有 1,001 位；代码给出两种分母下的结果。 是否选择某门槛，要看希望覆盖哪些用户以及指标是否需要更多记录支持。 不能把未被采集到的行为当成零，也不能把这两个组直接命名为稳定的高/低活跃人群。</p><pre><code class="language-python"># 重点：只改变记录门槛；用户占比和记录贡献占比分别保留全部用户、全部帖子的分母。
# 建立列表，依次保存两个记录门槛下的汇总结果。
q1_rows = []
# 分别使用 3 条和 5 条记录的门槛；其他窗口、用户表和分母保持一致。
for q1_threshold in [3, 5]:
    # 在同一用户表中筛选达到当前门槛的用户，保留原来的统计窗口。
    q1_selected = users.loc[users[&#x27;n_posts&#x27;].ge(q1_threshold)]
    # 将当前一轮或一个分组的结果字典追加到列表，随后转换为 DataFrame。
    q1_rows.append({
        &#x27;记录门槛&#x27;: q1_threshold,
        &#x27;用户数&#x27;: len(q1_selected),
        &#x27;用户占比&#x27;: len(q1_selected) / len(users),  # 分母为全部已观测用户数。
        &#x27;记录贡献数&#x27;: q1_selected[&#x27;n_posts&#x27;].sum(),
        &#x27;记录贡献占比&#x27;: q1_selected[&#x27;n_posts&#x27;].sum() / len(posts),  # 分母为全部帖子记录数。
    })
# 每个字典成为结果表的一行，便于比较两个门槛的人数和记录贡献。
q1_comparison = pd.DataFrame(q1_rows)
# 在 Notebook 输出区显示本步的结果表。
display(q1_comparison)</code></pre></details>

<a id="section-7-2"></a>
### 7.2 开放问题：“更常谈饮食”会不会只是词表定义的结果？ **[练习]**

在现有饮食词表中加入“外卖”和“咖啡”，帖子命中比例和用户等权比例各改变多少？
新增命中的正文是否都在表达作者自己的饮食偏好？
你会展示哪些记录，说明新规则的收益与误检可能？

**在下面的空白代码格写自己的分析；参考答案在其后折叠。**

<details><summary>展开 7.2 参考答案与代码</summary><p><strong>参考思路：</strong> 先保持数据范围和分母不变，只改变词表，才能把变化归因于这次规则修改。 两种权重都要看，随后阅读新增命中的文本。新增词会增加或保持命中数， 但是否提高识别质量，需要语境核验，不能从命中数上升直接判断。</p><p>参考代码同时比较全体已观测用户与固定的至少 3 条记录用户集合。 小样本语境检查只用于发现问题，不用于宣称准确率。 “咖啡”可能出现在广告、引用或工作语境中，因此结论仍应表述为“饮食词命中比例”。</p><pre><code class="language-python"># 重点：固定样本与用户门槛，只改变词表，才能比较规则变化带来的影响。
# 保存原饮食词表的逐帖命中结果，作为比较基线。
q2_old = topic_flags[&#x27;饮食&#x27;]
# 用 + 构造扩展词表，不向原 TOPICS 列表原地追加词语。
q2_words = TOPICS[&#x27;饮食&#x27;] + [&#x27;外卖&#x27;, &#x27;咖啡&#x27;]
# 对同一批帖子重新运行扩展词表，只改变规则、不改变样本。
q2_new = keyword_hit(posts[&#x27;text&#x27;], q2_words)
# 按用户聚合原规则命中比例，分母为该用户全部已采集记录。
q2_old_users = q2_old.groupby(posts[&#x27;user_id&#x27;]).mean()
# 同样按用户聚合新规则，让新旧结果使用相同用户与相同记录分母。
q2_new_users = q2_new.groupby(posts[&#x27;user_id&#x27;]).mean()
# 并列比较帖子等权、全体用户等权和固定门槛用户等权三种口径。
q2_comparison = pd.DataFrame({
    &#x27;原词表&#x27;: [q2_old.mean(), q2_old_users.mean(), q2_old_users.loc[eligible_ids].mean()],
    &#x27;扩展词表&#x27;: [q2_new.mean(), q2_new_users.mean(), q2_new_users.loc[eligible_ids].mean()],
}, index=[&#x27;全部帖子等权比例&#x27;, &#x27;全体用户等权比例&#x27;, &#x27;固定至少3条记录用户等权比例&#x27;])
# 新比例减旧比例再乘 100，得到百分点变化，而不是相对增长百分比。
q2_comparison[&#x27;变化百分点&#x27;] = (q2_comparison[&#x27;扩展词表&#x27;] - q2_comparison[&#x27;原词表&#x27;]) * 100
# 在 Notebook 输出区显示本步的结果表。
display(q2_comparison)
# 打印计算结果，核对数量、分母、阈值或运行状态。
print(&#x27;新增命中记录数：&#x27;, (q2_new &amp; ~q2_old).sum())
# q2_new &amp; ~q2_old 只选新增命中记录，取五条检查增加词语是否符合主题。
q2_examples = posts.loc[q2_new &amp; ~q2_old, [&#x27;city&#x27;, &#x27;text&#x27;]].head(5).copy()
# 只截短核验样例的展示副本，原表和新旧词表统计不变。
q2_examples[&#x27;text&#x27;] = q2_examples[&#x27;text&#x27;].str.slice(0, 150)
# 在 Notebook 输出区显示本步的结果表。
display(q2_examples)</code></pre></details>

<a id="section-7-3"></a>
### 7.3 开放问题：怎样定义“负向表达更多”？ **[练习]**

“仅负向词命中”与“含负向词（包括正负混合）”是否给出相同结果？
分母用全部帖子，还是只用命中过情绪词的帖子，会有什么区别？
能否据此判断用户更不幸福或更抑郁？如果不能，你会怎样写结论？

**在下面的空白代码格写自己的比较；参考答案在其后折叠。**

<details><summary>展开 7.3 参考答案与代码</summary><p><strong>参考思路：</strong> 先把“负向”的定义说清楚。正负混合记录是否算入，会改变分子。 “全部帖子”与“词表覆盖到的帖子”是两个不同的分母；只用命中帖子会排除未命中记录。 也可报告用户等权结果，避免把高记录用户的表达当成所有用户的表达。</p><p>无论采用哪种定义，这些结果只是词表对已采集文本的匹配，不是用户心理状态的测量。 否定、转述、反讽与缺失上下文都可能造成误判。源模型标签也没有解决采集覆盖或心理测量问题。</p><p>可用的结论句式：<strong>“在四川两周采集范围内，按这份词表及这一定义，____比例为____； 其中词表覆盖比例为____。该结果描述公开文本中的规则命中，不能推出用户的幸福感或心理健康状态。”</strong></p><pre><code class="language-python"># 重点：仅负向命中与包含任意负向词是两种定义；全部帖子与情绪词命中帖子也是两种分母。
# | 是逐行逻辑或，标记命中任意正向或负向词的记录。
q3_matched = positive_hit | negative_hit
# 只保留“仅负向命中”类别，排除正负同时命中的记录。
q3_negative_only = emotion_labels.eq(&#x27;仅负向命中&#x27;)
# 含任意负向词的记录也包括混合命中，定义比“仅负向”更宽。
q3_any_negative = negative_hit
# 统计情绪词命中记录数，作为条件比例的分母；它不同于全部帖子数。
q3_denominator = int(q3_matched.sum())
# 逐行保存两种负向定义下的计数和不同分母的比例。
q3_rows = []
# 对两种负向定义分别统计，并同时展示帖子等权与用户等权结果。
for q3_name, q3_flag in [(&#x27;仅负向词命中&#x27;, q3_negative_only), (&#x27;含任意负向词&#x27;, q3_any_negative)]:
    # 先对每位用户求记录命中比例，再对用户等权平均，避免高发帖用户贡献更多权重。
    q3_user_rates = q3_flag.groupby(posts[&#x27;user_id&#x27;]).mean()
    # 将当前一轮或一个分组的结果字典追加到列表，随后转换为 DataFrame。
    q3_rows.append({
        &#x27;定义&#x27;: q3_name,
        &#x27;命中帖子数&#x27;: int(q3_flag.sum()),
        &#x27;占全部帖子比例&#x27;: q3_flag.mean(),  # 布尔均值的分母为所有帖子，未命中记录也计入。
        &#x27;占情绪词命中帖子比例&#x27;: q3_flag.sum() / q3_denominator if q3_denominator else np.nan,  # 分母仅为情绪词命中记录；为 0 时返回缺失，避免除零。
        &#x27;全体用户等权比例&#x27;: q3_user_rates.mean(),  # 对各用户的记录比例求均值，不对用户先按记录数加权。
    })
# 在 Notebook 输出区显示本步的结果表。
display(pd.DataFrame(q3_rows))
# 打印计算结果，核对数量、分母、阈值或运行状态。
print(&#x27;词表覆盖全部帖子的比例：&#x27;, q3_matched.mean())
# 打印计算结果，核对数量、分母、阈值或运行状态。
print(&#x27;未命中帖子数：&#x27;, (~q3_matched).sum())</code></pre></details>

<a id="section-8"></a>
## 8 分析流程

写清研究问题 → 核对数据是否保留所需行为 → 确定帖子或用户等分析单位 → 定义指标和分母 →
查看表格与图 → 阅读真实语境 → 修改一次定义并比较 → 写出范围明确的结论。

本教程重点是描述与解释，没有因果估计、统计显著性检验或用户心理诊断。
如果进一步做统计推断，还需要根据用户重复记录、采集方式和比较目标设计分析。

读表、筛选与分组操作可查阅 [Pandas Analysis](02_pandas_analysis.ipynb)，
绘图操作可查阅 [Plotting](03_plotting.ipynb)。